# Предсказание калорийности блюд

## Этап 3. Обучение модели

В этом ноутбуке я обучаю мультимодальную модель из `scripts/` и проверяю, получается ли MAE меньше 50 ккал на тестовой выборке. Обучение запускаю в Google Colab на GPU.

## 1. Подготовка окружения

Клонирую репозиторий с кодом, перехожу в его папку и ставлю зависимости. В Colab уже стоит torch с поддержкой CUDA, так что pip его не переустанавливает.

In [ ]:
![ -d Nutrition ] || git clone https://github.com/andr3w13/Nutrition.git
%cd Nutrition
%pip install -q -r requirements.txt

Датасет скачиваю с Google Drive так же, как в ноутбуке с EDA. Если папка `data/` уже есть, повторно ничего не качается.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import pandas as pd
import torch

GDRIVE_FILE_ID = "19cIhxmvhRVpRqGHnBCbxLEfEZ51jvsTG"
ARCHIVE = Path("nutrition.zip")
DATA_DIR = Path("data")

if not (DATA_DIR / "dish.csv").exists():
    if not ARCHIVE.exists():
        gdown.download(id=GDRIVE_FILE_ID, output=str(ARCHIVE), quiet=False)
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(".")

print("Данные в папке", DATA_DIR.resolve())

In [ ]:
print("CUDA доступна", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Конфиг

Все параметры обучения лежат в `scripts/config.py`, ниже вывожу их, чтобы было видно, с какими настройками шло обучение.

Предсказание калорийности это регрессия, поэтому голова модели заканчивается одним нейроном `nn.Linear(HIDDEN_DIM, 1)` без функции активации, а в качестве лосса стоит `nn.L1Loss`, то есть напрямую оптимизируется MAE. Размер батча маленький (`BATCH_SIZE = 4`), чтобы не упереться в память GPU, ведь последние слои обоих энкодеров разморожены.

In [ ]:
from scripts.config import Config

pd.Series({k: v for k, v in vars(Config).items() if not k.startswith("_")}, dtype=object)

## 3. Обучение

Функция `train` фиксирует сид, обучает модель и после каждой эпохи печатает train loss и метрики на валидации. Веса лучшей по val MAE эпохи сохраняются по пути `Config.SAVE_PATH`.

In [ ]:
from scripts.utils import train

model, history = train(Config)

In [ ]:
print(model.classifier)

In [ ]:
history_df = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_df["epoch"], history_df["train_loss"], color="#2a78d6", linewidth=2, label="train loss (MAE)")
ax.plot(history_df["epoch"], history_df["val_mae"], color="#eb6834", linewidth=2, label="val MAE")
ax.axhline(50, color="gray", linestyle="--", linewidth=1, label="цель 50 ккал")
ax.set(xlabel="эпоха", ylabel="ккал", title="Кривые обучения")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

history_df.round(2)

## 4. Проверка на тестовой выборке

Загружаю сохранённые веса по пути из конфига и считаю метрики на test, который в обучении и выборе эпохи не участвовал.

In [ ]:
from transformers import AutoTokenizer

from scripts.utils import MultimodalModel, evaluate, get_dataloaders

device = torch.device(Config.DEVICE if torch.cuda.is_available() else "cpu")
best_model = MultimodalModel(Config).to(device)
best_model.load_state_dict(torch.load(Config.SAVE_PATH, map_location=device))

tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL_NAME)
_, _, test_loader = get_dataloaders(Config, tokenizer)

test_metrics = evaluate(best_model, test_loader, Config, device)
print(f"Test MAE {test_metrics['mae']:.2f} | Test RMSE {test_metrics['rmse']:.2f}")
print("Цель MAE < 50 достигнута" if test_metrics["mae"] < 50 else "Цель MAE < 50 не достигнута")

## 5. Сохранение артефакта

Веса лучшей модели функция `train` уже сохранила в `Config.SAVE_PATH`. Файлы в Colab удаляются после отключения сессии, поэтому копирую артефакт ещё и на Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/nutrition_models"
os.makedirs(save_dir, exist_ok=True)
shutil.copy(Config.SAVE_PATH, save_dir)
print("Артефакт скопирован в", save_dir)